In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ajoyprasad1998/qwen-3b/adapter_model.safetensors
/kaggle/input/datasets/ajoyprasad1998/qwen-3b/adapter_config.json
/kaggle/input/datasets/ajoyprasad1998/qwen-3b/README.md
/kaggle/input/datasets/ajoyprasad1998/qwen-3b/tokenizer.json
/kaggle/input/datasets/ajoyprasad1998/qwen-3b/tokenizer_config.json
/kaggle/input/datasets/ajoyprasad1998/qwen-3b/chat_template.jinja
/kaggle/input/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/medquad.csv


In [2]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 12.2 MB/s eta 0:00:0000:010:01


In [3]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 33.7 MB/s eta 0:00:00a 0:00:01


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, DataCollatorForLanguageModeling, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
model_id = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [7]:
import pandas as pd

In [8]:
df=pd.read_csv("/kaggle/input/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/medquad.csv")

In [9]:
df

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
...,...,...,...,...
16407,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16408,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16409,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16410,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...


In [10]:
df = df[['question', 'answer']].dropna()

In [11]:
print("After cleaning:", df.shape)

After cleaning: (16407, 2)


In [12]:
from datasets import load_dataset, Dataset

In [13]:
dataset = Dataset.from_pandas(df)

In [14]:
def format_chat(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat)
dataset = dataset.train_test_split(test_size=0.1)
print(dataset)

Map:   0%|          | 0/16407 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', '__index_level_0__', 'text'],
        num_rows: 14766
    })
    test: Dataset({
        features: ['question', 'answer', '__index_level_0__', 'text'],
        num_rows: 1641
    })
})


In [15]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.1 MB/s eta 0:00:00:00:0100:01


In [16]:
from transformers import DataCollatorForLanguageModeling, Trainer

MAX_SEQ_LEN = 1024
tokenizer.model_max_length = MAX_SEQ_LEN
batch_size = 2
grad_accum = 4
num_epochs = 3
warmup_steps = 166

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_tensors=None,
    )

tokenized_train = dataset["train"].map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_test = dataset["test"].map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./qwen2.5-lora-output",
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=num_epochs,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    learning_rate=2e-4,
    fp16=True,
    optim="adamw_torch",
    max_grad_norm=0.3,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

Map:   0%|          | 0/14766 [00:00<?, ? examples/s]

Map:   0%|          | 0/1641 [00:00<?, ? examples/s]

In [17]:
trainer.train()

Step,Training Loss
10,2.058469
20,1.610593
30,1.347538
40,1.116237
50,1.147587
60,1.025406
70,1.036349
80,0.994835
90,0.946559
100,0.973567


TrainOutput(global_step=5538, training_loss=0.7227353950664428, metrics={'train_runtime': 29516.5677, 'train_samples_per_second': 1.501, 'train_steps_per_second': 0.188, 'total_flos': 3.044719035710669e+17, 'train_loss': 0.7227353950664428, 'epoch': 3.0})

In [18]:
model.save_pretrained("./qwen-3b-lora-medquad-adapter")
tokenizer.save_pretrained("./qwen-3b-lora-medquad-adapter")

('./qwen-3b-lora-medquad-adapter/tokenizer_config.json',
 './qwen-3b-lora-medquad-adapter/chat_template.jinja',
 './qwen-3b-lora-medquad-adapter/tokenizer.json')

In [19]:
import zipfile
import os

def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=folder_path)
                zipf.write(file_path, arcname)

folder_to_zip = "/kaggle/working/qwen-3b-lora-medquad-adapter"
zip_file_name = "/kaggle/working/qwen-3b-lora-medquad-adapter.zip"
zip_folder(folder_to_zip, zip_file_name)

In [17]:
!pip install -q rouge-score bert-score pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.8 MB/s eta 0:00:00


**FINE TUNED MODEL LOADING**

In [18]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import os

MODEL_PATH = "/kaggle/input/datasets/ajoyprasad1998/qwen-3b"

is_adapter = os.path.exists(os.path.join(MODEL_PATH, "adapter_config.json"))

if is_adapter:
    base_model_name = "Qwen/Qwen2.5-3B-Instruct"
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Loading LoRA adapter...")
    model = PeftModel.from_pretrained(base_model, MODEL_PATH)
else:
    print("Loading merged model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

model.eval()

test_dataset = dataset["test"]

test_questions = [item["question"] for item in test_dataset]
test_references = [item["answer"] for item in test_dataset]

print(f"Test set size: {len(test_questions)}")

Loading base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading LoRA adapter...
Test set size: 1641


In [19]:
def generate_batch(questions, model, tokenizer, batch_size=16, max_new_tokens=300, temperature=0.7, do_sample=True, top_p=0.9):
    predictions = []
    total_batches = (len(questions) + batch_size - 1) // batch_size
    print(f"Generating predictions in {total_batches} batches (batch_size={batch_size})")
    for i in tqdm(range(0, len(questions), batch_size)):
        batch_questions = questions[i:i+batch_size]
        formatted_batch = []
        for q in batch_questions:
            messages = [{"role": "user", "content": q}]
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            formatted_batch.append(formatted)
        inputs = tokenizer(
            formatted_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        for j, output in enumerate(outputs):
            response = tokenizer.decode(output, skip_special_tokens=True)
            response = response[len(formatted_batch[j]):].strip()
            predictions.append(response)
    return predictions

predictions = generate_batch(test_questions, model, tokenizer, batch_size=16, max_new_tokens=300, temperature=0.7, do_sample=True, top_p=0.9)

Generating predictions in 103 batches (batch_size=16)


100%|██████████| 103/103 [1:09:32<00:00, 40.51s/it]


In [20]:
!pip install -q rouge-score bert-score pandas tqdm

****BASE MODEL LOADING****

In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

model_id = "Qwen/Qwen2.5-3B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model.eval()

test_dataset = dataset["test"]
test_questions = [item["question"] for item in test_dataset]
test_references = [item["answer"] for item in test_dataset]

print(f"Test set size: {len(test_questions)}")

def generate_batch(questions, model, tokenizer, batch_size=16, max_new_tokens=100, temperature=0.7, do_sample=True, top_p=0.9):
    predictions = []
    total_batches = (len(questions) + batch_size - 1) // batch_size
    print(f"Generating predictions in {total_batches} batches (batch_size={batch_size})")
    for i in tqdm(range(0, len(questions), batch_size)):
        batch_questions = questions[i:i+batch_size]
        formatted_batch = []
        for q in batch_questions:
            messages = [{"role": "user", "content": q}]
            formatted = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            formatted_batch.append(formatted)
        inputs = tokenizer(
            formatted_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        for j, output in enumerate(outputs):
            response = tokenizer.decode(output, skip_special_tokens=True)
            response = response[len(formatted_batch[j]):].strip()
            predictions.append(response)
    return predictions

base_predictions_3b = generate_batch(test_questions, base_model, tokenizer, batch_size=16, max_new_tokens=100, temperature=0.7, do_sample=True, top_p=0.9)

print(f"Generated {len(base_predictions_3b)} predictions")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Test set size: 1641
Generating predictions in 103 batches (batch_size=16)


100%|██████████| 103/103 [13:18<00:00,  7.75s/it]

Generated 1641 predictions


In [25]:
import numpy as np
from rouge_score import rouge_scorer


ft_predictions_3b =predictions


# Compute ROUGE for base model (3B)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_base = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for pred, ref in zip(base_predictions_3b, test_references):
    try:
        scores = scorer.score(pred, ref)
        for key in rouge_base:
            rouge_base[key].append(scores[key].fmeasure)
    except:
        continue

rouge_base_stats = {}
for key in rouge_base:
    if rouge_base[key]:
        rouge_base_stats[key] = {'mean': np.mean(rouge_base[key]), 'std': np.std(rouge_base[key])}
    else:
        rouge_base_stats[key] = {'mean': 0, 'std': 0}

# Compute ROUGE for fine-tuned model (3B)
rouge_ft = {'rouge1': [], 'rouge2': [], 'rougeL': []}
for pred, ref in zip(ft_predictions_3b, test_references):
    try:
        scores = scorer.score(pred, ref)
        for key in rouge_ft:
            rouge_ft[key].append(scores[key].fmeasure)
    except:
        continue

rouge_ft_stats = {}
for key in rouge_ft:
    if rouge_ft[key]:
        rouge_ft_stats[key] = {'mean': np.mean(rouge_ft[key]), 'std': np.std(rouge_ft[key])}
    else:
        rouge_ft_stats[key] = {'mean': 0, 'std': 0}

# Compute Exact Match
exact_base = sum([p.strip().lower() == r.strip().lower() for p, r in zip(base_predictions_3b, test_references)])
exact_base_pct = exact_base / len(base_predictions_3b) * 100

exact_ft = sum([p.strip().lower() == r.strip().lower() for p, r in zip(ft_predictions_3b, test_references)])
exact_ft_pct = exact_ft / len(ft_predictions_3b) * 100

# Print comparison
print("\n" + "="*70)
print("ROUGE COMPARISON: Qwen2.5-3B Base vs Fine-Tuned")
print("="*70)
print(f"\n{'Metric':<15} {'Base':<20} {'Fine-Tuned':<20} {'Improvement':<15}")
print("-"*70)

for metric in ['rouge1', 'rouge2', 'rougeL']:
    base_val = rouge_base_stats[metric]['mean']
    ft_val = rouge_ft_stats[metric]['mean']
    improvement = ((ft_val - base_val) / base_val) * 100 if base_val != 0 else 0
    arrow = "↑" if improvement > 0 else "↓"
    print(f"{metric.upper():<15} {base_val:<20.4f} {ft_val:<20.4f} {arrow} {improvement:+.1f}%")

print(f"\n{'Exact Match':<15} {exact_base_pct:<20.2f}% {exact_ft_pct:<20.2f}% {'':<15}")

print("\n" + "="*70)


ROUGE COMPARISON: Qwen2.5-3B Base vs Fine-Tuned

Metric          Base                 Fine-Tuned           Improvement    
----------------------------------------------------------------------
ROUGE1          0.1904               0.3504               ↑ +84.1%
ROUGE2          0.0455               0.1982               ↑ +335.5%
ROUGEL          0.1160               0.2543               ↑ +119.3%

Exact Match     0.00                % 0.00                %                

